# M7 Run 1 - clean baseline (single split) + resolution A/B + non-deep control

**Not the honest OOF number** (that is Run 2). Single split: train folds 1-7 / val fold 8 (early-stop on **loss**) / test fold 9 (**full, realistic prevalence**). fold 10 NEVER touched.

This run answers: (1) does the clean pipeline hold and at what level? (2) which **resolution** {1024/2048/5000}? (3) **cost calibration** + **pipeline-sanity floor** via a non-deep control (XGBoost on flattened raw signal, capped L<=2048).

> The **deep-vs-non-deep verdict** is read at **Run 7** on the tuned config, with matched variance reduction. At Run 1 the control is sanity/cost only.

> Prereq: **PyTorch** (CPU) + **xgboost**. If torch missing: `pip install torch --index-url https://download.pytorch.org/whl/cpu`

### Block 1.0: Setup and configuration

In [ ]:
# M7 Run 1 - clean baseline (single split) + resolution A/B + non-deep control.
# Protocol: train folds 1-7 / val fold 8 (early-stop on LOSS) / test fold 9 (FULL, realistic prevalence).
# fold 10 NEVER touched. Honest OOF number comes at Run 2; deep-vs-non-deep VERDICT comes at Run 7.
import os, sys, json, time, warnings
import numpy as np, pandas as pd
from scipy.signal import butter, sosfiltfilt, resample
from sklearn.metrics import average_precision_score, roc_auc_score
from joblib import Parallel, delayed
from tqdm import tqdm
warnings.filterwarnings('ignore')

ROOT=r'.'
PROCESSED=os.path.join(ROOT,'data','processed'); SRC=os.path.join(ROOT,'src')
FIG=os.path.join(ROOT,'reports','figures'); METRICS=os.path.join(ROOT,'reports','metrics')
MODELS=os.path.join(ROOT,'models','M7_run1'); CACHE_DIR=os.path.join(PROCESSED,'m7_run1_cache')
for d in (FIG,METRICS,MODELS,CACHE_DIR): os.makedirs(d,exist_ok=True)
sys.path.insert(0,SRC)
from signal_loading import load_signal, LEADS_CANONICAL
from evaluation import evaluate_standard, _boot_ci

with open(os.path.join(PROCESSED,'filter_config.json'),encoding='utf-8') as f:
    FCFG=json.load(f)['filter_FINAL']
FS=FCFG['fs']; LEADS=list(LEADS_CANONICAL); LEAD_IDX={L:LEADS_CANONICAL.index(L) for L in LEADS}
SOS=butter(FCFG['order'],[FCFG['low']/(FS/2),FCFG['high']/(FS/2)],btype='band',output='sos')
def bp(x): return sosfiltfilt(SOS,np.asarray(x,dtype=np.float64))

# ---- Run 1 config (single source of truth) ----
RANDOM_STATE=42
RESOLUTIONS=[1024,2048,5000]      # A/B on the resampled length (12 x L)
CONTROL_RES=[1024,2048]           # non-deep control capped <=2048 (never native 5000 = 60k feats)
N_NEG_TRAIN=12000                 # subsampled non-WPW pool for training (balanced batches draw from it)
SEEDS=[0,1,2,3,4]                 # 5-seed ensemble
EPOCHS=60; PATIENCE=10; BATCH=64; LR=1e-3; WD=1e-4
FOCAL_GAMMA=2.0; FOCAL_ALPHA=0.5
N_JOBS=6                          # CPU-only machine: n_jobs<=6, no CUDA

meta=pd.read_csv(os.path.join(PROCESSED,'metadata_combined.csv'),dtype={'ecg_id':str})
def _wpw(fa,fb=None):
    m=(meta.fold==fa)&(meta.label==1) if fb is None else (meta.fold.between(fa,fb))&(meta.label==1)
    return int(meta[m].shape[0])
print('M7 Run 1 | resolutions',RESOLUTIONS,'| control',CONTROL_RES,'| seeds',SEEDS)
print('WPW: train(1-7) %d | val fold8 %d | test fold9 %d | fold10 %d [UNTOUCHED]'%(_wpw(1,7),_wpw(8),_wpw(9),_wpw(10)))
print('Block 1.0 OK.')

### Block 1.1: Data caching (native 5000 Hz, band-pass filtered) - chunk-to-disk

In [ ]:
# Train = folds 1-7 (ALL WPW + N_NEG_TRAIN subsampled non-WPW). Val = fold 8 (full). Test = fold 9 (full).
# Workers write chunks to disk then we concat (no big arrays via joblib -> avoids WinError 1450). float32.
# Robust load: a few WFDB headers are malformed (bad base_date) -> skip & log (project convention: extraction_failed).
def _build_rows(df):
    return list(zip(df.ecg_id.tolist(), df.source.tolist(), df.label.values.astype('float32').tolist()))

def _extract_chunk(rows, outpath):
    xs=[]; ys=[]; failed=[]
    for eid,src,lab in rows:
        try:
            raw=load_signal(eid,src)                                  # (5000,12) mV
            x=np.stack([bp(raw[:,LEAD_IDX[Ld]]) for Ld in LEADS])     # (12,5000) filtered
            xs.append(x.astype('float32')); ys.append(lab)
        except Exception:
            failed.append((eid,src,lab))                              # malformed header etc. -> skip (keep alignment)
    arr=np.stack(xs).astype('float32') if xs else np.empty((0,12,5000),dtype='float32')
    np.save(outpath,arr)
    return outpath, ys, failed

def _extract_set(rows, tag):
    parts=[p for p in np.array_split(np.arange(len(rows)),N_JOBS) if len(p)]
    paths=[os.path.join(CACHE_DIR,'%s_chunk%d.npy'%(tag,j)) for j in range(len(parts))]
    res=Parallel(n_jobs=N_JOBS)(delayed(_extract_chunk)([rows[i] for i in part],p) for part,p in zip(parts,paths))
    X=np.concatenate([np.load(pth) for pth,_,_ in res],axis=0).astype('float32')   # part order preserved
    y=np.array([lab for _,ys,_ in res for lab in ys],dtype='float32')
    failed=[f for _,_,fl in res for f in fl]
    for pth,_,_ in res:
        if os.path.exists(pth): os.remove(pth)
    if failed:
        npos=int(sum(1 for _,_,lab in failed if lab==1))
        print('  [%s] %d records skipped (malformed header), of which %d WPW'%(tag,len(failed),npos))
    return X,y

CACHE=os.path.join(CACHE_DIR,'m7_run1_native.npz')
if os.path.exists(CACHE):
    z=np.load(CACHE)
    Xtr_native,ytr=z['Xtr'],z['ytr']; Xval_native,yval=z['Xval'],z['yval']; Xte_native,yte=z['Xte'],z['yte']
    print('[guard] native cache reloaded')
else:
    tr=meta[meta.fold.between(1,7)]; wpw=tr[tr.label==1]
    neg=tr[tr.label==0].sample(min(N_NEG_TRAIN,int((tr.label==0).sum())),random_state=RANDOM_STATE)
    trm=pd.concat([wpw,neg]).sample(frac=1,random_state=RANDOM_STATE).reset_index(drop=True)
    valm=meta[meta.fold==8].reset_index(drop=True); tem=meta[meta.fold==9].reset_index(drop=True)
    print('extracting train %d (%d WPW) | val %d (%d WPW) | test %d (%d WPW)...'%(
        len(trm),int((trm.label==1).sum()),len(valm),int((valm.label==1).sum()),len(tem),int((tem.label==1).sum())))
    rtr=_build_rows(trm); rval=_build_rows(valm); rte=_build_rows(tem)
    Xtr_native,ytr=_extract_set(rtr,'train'); Xval_native,yval=_extract_set(rval,'val'); Xte_native,yte=_extract_set(rte,'test')
    np.savez_compressed(CACHE,Xtr=Xtr_native,ytr=ytr,Xval=Xval_native,yval=yval,Xte=Xte_native,yte=yte)
    print('[guard] native cache written')
print('native: Xtr %s (%d WPW) | Xval %s (%d WPW) | Xte %s (%d WPW)'%(
    Xtr_native.shape,int(ytr.sum()),Xval_native.shape,int(yval.sum()),Xte_native.shape,int(yte.sum())))

def to_resolution(Xnative,L):
    X = Xnative.astype('float32',copy=True) if L==Xnative.shape[2] else resample(Xnative,L,axis=2).astype('float32')
    X=(X-X.mean(2,keepdims=True))/(X.std(2,keepdims=True)+1e-6)        # per-lead z-score at this resolution
    return np.nan_to_num(X).astype('float32')
print('Block 1.1 OK.')

### Block 1.2: 1D-ResNet (modest) - identical across resolutions

In [ ]:
# 1D-ResNet (modest) - identical architecture across all resolutions (AdaptiveMaxPool handles any L).
import torch, torch.nn as nn
torch.set_num_threads(N_JOBS)

class BasicBlock1d(nn.Module):
    def __init__(self,cin,cout,stride=1):
        super().__init__()
        self.conv1=nn.Conv1d(cin,cout,7,stride=stride,padding=3,bias=False); self.bn1=nn.BatchNorm1d(cout)
        self.conv2=nn.Conv1d(cout,cout,7,padding=3,bias=False); self.bn2=nn.BatchNorm1d(cout)
        self.relu=nn.ReLU(inplace=True); self.down=None
        if stride!=1 or cin!=cout:
            self.down=nn.Sequential(nn.Conv1d(cin,cout,1,stride=stride,bias=False),nn.BatchNorm1d(cout))
    def forward(self,x):
        idt=x if self.down is None else self.down(x)
        o=self.relu(self.bn1(self.conv1(x))); o=self.bn2(self.conv2(o)); return self.relu(o+idt)

class ResNet1d(nn.Module):
    def __init__(self,ch=(16,32,64),pdrop=0.3):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv1d(12,ch[0],15,stride=2,padding=7,bias=False),
                                nn.BatchNorm1d(ch[0]),nn.ReLU(inplace=True),nn.MaxPool1d(4))
        self.layer1=BasicBlock1d(ch[0],ch[0],1)
        self.layer2=BasicBlock1d(ch[0],ch[1],2)
        self.layer3=BasicBlock1d(ch[1],ch[2],2)
        self.pool=nn.AdaptiveMaxPool1d(1)
        self.head=nn.Sequential(nn.Flatten(),nn.Dropout(pdrop),nn.Linear(ch[2],1))
    def forward(self,x):
        return self.head(self.pool(self.layer3(self.layer2(self.layer1(self.stem(x)))))).squeeze(1)
print('Block 1.2 OK | params:', sum(p.numel() for p in ResNet1d().parameters()))

### Block 1.3: Focal loss, augmentation, balanced-batch training (early-stop on val loss)

In [ ]:
# Focal loss + rich augmentation + balanced-batch training; early-stop on val LOSS (not the ~13-WPW AP).
def focal_loss(logits,targets,gamma=FOCAL_GAMMA,alpha=FOCAL_ALPHA):
    ce=nn.functional.binary_cross_entropy_with_logits(logits,targets,reduction='none')
    p=torch.sigmoid(logits); pt=torch.where(targets==1,p,1-p)
    a=torch.where(targets==1,torch.full_like(targets,alpha),torch.full_like(targets,1-alpha))
    return (a*(1-pt).pow(gamma)*ce).mean()

def augment(xb):
    T=xb.shape[2]; mx=max(1,int(0.04*T))
    xb=torch.roll(xb,int(torch.randint(-mx,mx+1,(1,)).item()),dims=2)        # time shift
    xb=xb*(0.8+0.4*torch.rand(xb.shape[0],1,1))                              # amplitude scale
    xb=xb+0.02*torch.randn_like(xb)                                          # gaussian noise
    t=torch.linspace(0,1,T).view(1,1,-1)                                     # baseline wander
    fb=0.5+2.0*torch.rand(xb.shape[0],1,1); ph=2*np.pi*torch.rand(xb.shape[0],1,1)
    xb=xb+0.05*torch.sin(2*np.pi*fb*t+ph)
    if torch.rand(1).item()<0.3:                                            # lead dropout
        xb[:,int(torch.randint(0,12,(1,)).item()),:]=0.0
    return xb

def predict(model,X):
    model.eval(); out=[]
    with torch.no_grad():
        for i in range(0,len(X),256):
            out.append(torch.sigmoid(model(torch.tensor(X[i:i+256]))).numpy())
    return np.nan_to_num(np.concatenate(out))

def train_one(seed,Xtr,ytr,Xval,yval):
    torch.manual_seed(seed); np.random.seed(seed); rng=np.random.default_rng(seed)
    pos=np.where(ytr==1)[0]; neg=np.where(ytr==0)[0]; half=BATCH//2
    Xv=torch.tensor(Xval); yv=torch.tensor(yval)
    model=ResNet1d(); opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WD)
    steps=max(1,len(ytr)//BATCH); best=1e9; best_state=None; bad=0; last=0
    for ep in range(EPOCHS):
        last=ep+1; model.train()
        for _ in range(steps):
            bi=np.concatenate([rng.choice(pos,half,True),rng.choice(neg,half,True)])
            xb=augment(torch.tensor(Xtr[bi]).clone()); yb=torch.tensor(ytr[bi])
            opt.zero_grad(); loss=focal_loss(model(xb),yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
        model.eval()
        with torch.no_grad():
            vl=[focal_loss(model(Xv[i:i+256]),yv[i:i+256]).item() for i in range(0,len(Xv),256)]
        vloss=float(np.mean(vl))
        if vloss<best-1e-4: best=vloss; best_state={k:v.clone() for k,v in model.state_dict().items()}; bad=0
        else: bad+=1
        if bad>=PATIENCE: break
    model.load_state_dict(best_state); return model,last
print('Block 1.3 OK.')

### Block 1.4: Resolution A/B - 5-seed ensemble, evaluate_standard, gap, timing

In [ ]:
# Resolution A/B: train a 5-seed ensemble per resolution, evaluate_standard (fold9), gap, timing.
ab_rows=[]
for L in RESOLUTIONS:
    Xtr=to_resolution(Xtr_native,L); Xval=to_resolution(Xval_native,L); Xte=to_resolution(Xte_native,L)
    pv=[]; ptt=[]; ptr=[]; ap_seed=[]; auc_seed=[]; epochs=[]; t0=time.time()
    for s in SEEDS:
        model,ne=train_one(s,Xtr,ytr,Xval,yval); epochs.append(ne)
        pv.append(predict(model,Xval)); ptt.append(predict(model,Xte)); ptr.append(predict(model,Xtr))
        ap_seed.append(average_precision_score(yte,ptt[-1])); auc_seed.append(roc_auc_score(yte,ptt[-1]))
    Vens=np.mean(pv,0); Tens=np.mean(ptt,0); TRens=np.mean(ptr,0)
    ap_train=float(average_precision_score(ytr,TRens)); dt=(time.time()-t0)/60.0
    MULTI={'AP_mean':float(np.mean(ap_seed)),'AP_std':float(np.std(ap_seed)),
           'AUC_mean':float(np.mean(auc_seed)),'AUC_std':float(np.std(auc_seed))}
    m=evaluate_standard(name='M7run1_L%d'%L,y_oof=yval,score_oof=Vens,y_test=yte,score_test=Tens,
                        figures_dir=FIG,metrics_dir=METRICS,ap_train=ap_train,multiseed=MULTI,
                        extra={'run':'m7_run1','resolution':L,'mean_epochs':float(np.mean(epochs)),'train_min':dt})
    np.save(os.path.join(MODELS,'oof_val_L%d.npy'%L),Vens); np.save(os.path.join(MODELS,'score_test_L%d.npy'%L),Tens)
    ab_rows.append(dict(L=L,AP=m['AP'],AP_lo=m['AP_IC95'][0],AP_hi=m['AP_IC95'][1],AUC=m['AUC'],
                        AP_seed_mean=MULTI['AP_mean'],AP_seed_std=MULTI['AP_std'],gap=m['gap_train_test'],
                        epochs=float(np.mean(epochs)),minutes=dt))
    print('  [L=%d] ens AP %.3f CI[%.3f,%.3f] | per-seed %.3f+/-%.3f | gap %+.3f | %.0f ep | %.1f min'%(
        L,m['AP'],m['AP_IC95'][0],m['AP_IC95'][1],MULTI['AP_mean'],MULTI['AP_std'],m['gap_train_test'],np.mean(epochs),dt))
    del Xtr,Xval,Xte
ab=pd.DataFrame(ab_rows)
print('Block 1.4 OK.'); print(ab.to_string(index=False))

In [ ]:
# Non-deep control: XGBoost on flattened raw signal (12 x L), capped L<=2048.
# Run-1 role = COST calibration + pipeline-sanity floor ONLY. Deep-vs-non-deep VERDICT = Run 7,
# where the control receives variance reduction comparable to the CNN seed-ensemble.
import xgboost as xgb
ctrl_rows=[]
for L in CONTROL_RES:
    Xtr=to_resolution(Xtr_native,L).reshape(len(ytr),-1)
    Xte=to_resolution(Xte_native,L).reshape(len(yte),-1)
    t0=time.time()
    clf=xgb.XGBClassifier(n_estimators=400,max_depth=4,learning_rate=0.05,subsample=0.8,
                          colsample_bytree=0.3,n_jobs=N_JOBS,random_state=RANDOM_STATE,
                          eval_metric='aucpr',tree_method='hist')
    clf.fit(Xtr,ytr)
    p=clf.predict_proba(Xte)[:,1]; ap=float(average_precision_score(yte,p)); auc=float(roc_auc_score(yte,p))
    lo,hi=_boot_ci(yte,p,average_precision_score); dt=(time.time()-t0)/60.0
    deep=float(ab.loc[ab.L==L,'AP'].iloc[0]) if (ab.L==L).any() else float('nan')
    ctrl_rows.append(dict(L=L,n_feat=12*L,AP=ap,AP_lo=lo,AP_hi=hi,AUC=auc,deep_AP=deep,deep_minus_nondeep=deep-ap,fit_min=dt))
    print('  [control L=%d] XGBoost-raw AP %.3f CI[%.3f,%.3f] | deep %.3f | deep-nondeep %+.3f | %d feats | %.1f min'%(
        L,ap,lo,hi,deep,deep-ap,12*L,dt))
    del Xtr,Xte
ctrl=pd.DataFrame(ctrl_rows)
ctrl.to_csv(os.path.join(METRICS,'m7_run1_nondeep_control.csv'),index=False)
print('Block 1.5 OK.'); print(ctrl.to_string(index=False))

### Block 1.6: Run 1 summary - resolution pick + reminders

In [ ]:
# Run 1 summary: resolution pick (CI-overlap parsimony) + reminders (honest number = Run 2, verdict = Run 7).
ab_sorted=ab.sort_values('AP',ascending=False).reset_index(drop=True); best=ab_sorted.iloc[0]
overlap=ab[ab.AP_hi>=best.AP_lo].sort_values('L').reset_index(drop=True)   # cheapest resolution whose CI overlaps the best
pick=int(overlap.iloc[0].L)
summary={'run':'m7_run1','resolutions':RESOLUTIONS,'best_AP_resolution':int(best.L),'best_AP':float(best.AP),
         'parsimonious_pick':pick,'ab_table':ab.to_dict('records'),'control_table':ctrl.to_dict('records')}
with open(os.path.join(METRICS,'m7_run1_summary.json'),'w',encoding='utf-8') as f:
    json.dump(summary,f,ensure_ascii=False,indent=2,default=float)
print('=== M7 Run 1 - summary ===')
print(ab.to_string(index=False))
print('REF (AP OOF, NOT directly comparable to this 1-split fold9): M3 0.619 | M4 0.718 | vote 0.736')
print('best fold9 ensemble AP @ L=%d (%.3f); parsimonious pick (CI-overlap, cheapest) = L=%d'%(int(best.L),best.AP,pick))
print('Non-deep control (cost + sanity floor only at Run 1):')
print(ctrl.to_string(index=False))
print('--- reminders ---')
print(' - fold9 = 13 WPW, NOISY: sanity + resolution ballpark, NOT the honest number (Run 2 = OOF).')
print(' - deep-vs-non-deep VERDICT is read at Run 7 on the tuned config, matched variance reduction.')
print(' - sanity floor: if CNN << XGBoost-raw, suspect a pipeline bug, not a result.')
print('Block 1.6 OK.')